# Begin

Courtesy: https://docs.cleanrl.dev/rl-algorithms/dqn/#dqn_ataripy

In [1]:
# @launchit.collected

In [2]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import datetime
import json
import pprint
import re
import uuid
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Enum, IntEnum, StrEnum, auto # @launchit.collect
import multiprocessing as mp
import queue

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import Dataset, DataLoader
import torch.multiprocessing as torch_mp # not actually needed but keeped to be aligned with docs (see https://github.com/pytorch/pytorch/issues/70041)

import gymnasium as gym
import ale_py
import av

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect

from cleanrl.cleanrl_utils.atari_wrappers import (  # isort:skip
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)
from cleanrl.cleanrl_utils.buffers import ReplayBuffer

import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter
from logging_utils import *
from artifact_registry import *
from torch_helpers import *
import launchit 
import optuna_multiprocessing  # @launchit.collect
from hp_utils import *
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement

# Init

In [3]:
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()
    
def create_config():
    config = namedtuple('Config', 
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                        'self_fname, self_name, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, exec_mode, is_interactive')(
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        self_fname=None,
        self_name=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf)['jupyter_session']
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
            config = config._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)
    
    config = config._replace(is_interactive=config.exec_mode != ExecMode.LAUNCH_MODULE)    
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    return config

In [4]:
# @launchit.disable_2
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.17_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/17_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/17_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/17_rl',
 'self_fname': '/home/misha/dev/mine/neurolab/17_rl/17b_dqn_atari_mp_01.ipynb',
 'self_name': '17b_dqn_atari_mp_01',
 'subproject_name': '17_rl',
 'is_cuda': False,
 'cuda_device': 'cpu',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [5]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN = auto()
    WORKER = auto()

LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: LaunchGoal = lu.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED)
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    # System params
    random_seed: int = None
    torch_deterministic: bool = None

    # Environment params
    env_id: str = None
    envs_count: int = None # the number of parallel game environments

    # Agent params
    # ...

    # RL params
    gamma: float = None # the discount factor gamma

    # Training procedure params (DQN related) 
    global_steps_count: int = None
    warmup_steps_count: int = None # number of steps to run before the very first Agent's learn
    learn_steps_period: int = None  # number of steps to run between adjacent Agent's learnings
    oracle_steps_period: int = None # number of steps to run for Agent's weights are moved to Oracle (a-la stabilization period)
    start_e: float = None # the starting epsilon for exploration
    end_e: float = None # the ending epsilon for exploration
    exploration_fraction: float = None # the fraction of `total-timesteps` it takes from start_e to go end_e
    replay_buffer_size: int = None
    tau: float = None # the Orcale weights' update rate
    capture_video: str = None # video capture policy depending on rollouts, e.g. every(200)

    # Optimization params
    batch_size: int = None
    optimizer: str = None
    learn_rate: str = None

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()
HP.random_seed = 42
HP.torch_deterministic = True

# Launch

## LaunchState

In [6]:
# @launchit.disable_2
@dataclass(slots=True)
class LaunchState:
    mp_ctx: object = None
    env: list = None
    agent: object = None
    oracle: object = None
    policy_rollout_ipc: object = None
    workers: list = None
    capture_video_worker: object = None

## new_artifact_registry

In [7]:
# @launchit.disable_2
def new_artifact_registry(is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        mr = Mock()
        mr.register_model.return_value = 0
        return mr
        
    return ArtifactRegistry(CONFIG.model_group_uri)

## new_summary_writer

In [8]:
# @launchit.disable_2
def new_summary_writer(log_dir, is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        sw = Mock()
        sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
        return sw
    
    return RmqSummaryWriter(log_dir)

## Create

In [9]:
# @launchit.disable_2
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', HP.launch_id)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    random.seed(HP.random_seed)
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if HP.torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()
artifact_registry = new_artifact_registry()
artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    
meta = dict(
    optuna_trial_number=getattr(optuna_trial, 'number', None),
    hypers=HP._asdict(), 
    config=CONFIG._asdict(), 
)

with io.StringIO() as b:
    json.dump(meta, b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = lc.name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

LS = LaunchState()
LS.mp_ctx = torch_mp.get_context('spawn') # Spawn is needed for CUDA, fork doesn't work within PyTorch

Random seed=42
torch.backends.cudnn.deterministic=True
Tensorboard run=17b_dqn_atari_mp_01/0


# Environment

## create_env

In [10]:
def create_env(env_id, video_dir_name=None, random_seed=None, is_auto_reset=False):
    if video_dir_name is not None:
        env = gym.make(env_id, render_mode='rgb_array')
        env = gym.wrappers.RecordVideo(env, video_dir_name, episode_trigger=lambda episode_id: episode_id == 0)
    else:
        env = gym.make(env_id)
        
    env = gym.wrappers.RecordEpisodeStatistics(env)
    env = NoopResetEnv(env, noop_max=30)
    env = MaxAndSkipEnv(env, skip=4)
    env = EpisodicLifeEnv(env) 
    
    if 'FIRE' in env.unwrapped.get_action_meanings():
        env = FireResetEnv(env)

    if is_auto_reset:
        env = gym.wrappers.Autoreset(env)
        
    env = ClipRewardEnv(env)
    # Following three directly influence envs.single_observation_space
    env = gym.wrappers.ResizeObservation(env, (84, 84))
    env = gym.wrappers.GrayscaleObservation(env)
    env = gym.wrappers.FrameStackObservation(env, 4)

    if random_seed is not None:
        env.action_space.seed(random_seed) # req-d for random sampling from action space when there are multiple envs
        
    return env

## Configure 

In [11]:
# @launchit.disable
# @launchit.collect
HP.env_id = "BreakoutNoFrameskip-v4"
HP.envs_count = 8 # the number of parallel game environments
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': None,
 'global_steps_count': None,
 'warmup_steps_count': None,
 'learn_steps_period': None,
 'oracle_steps_period': None,
 'start_e': None,
 'end_e': None,
 'exploration_fraction': None,
 'replay_buffer_size': None,
 'tau': None,
 'capture_video': None,
 'batch_size': None,
 'optimizer': None,
 'learn_rate': None}


## Create

In [12]:
# @launchit.disable_2
LS.env = create_env(HP.env_id, random_seed=HP.random_seed)
assert isinstance(LS.env.action_space, gym.spaces.Discrete), "only discrete action space is supported"
LOG(f'{LS.env.observation_space=}, {LS.env.action_space=}, {LS.env.metadata=}')

LS.env.observation_space=Box(0, 255, (4, 84, 84), uint8), LS.env.action_space=Discrete(4), LS.env.metadata={'render_modes': ['human', 'rgb_array'], 'render_fps': 30}


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


# Agent

## Agent

In [13]:
class Agent(nn.Module):
    @dataclass(slots=True)
    class Params:
        d_model: int = 512
        actions_count: int = None 
    
    def __init__(self, params):
        super().__init__()
        self.params = params
        # TODO: use _layer_init
        self.network = nn.Sequential(
            nn.Conv2d(4, 32, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, self.params.d_model),
            nn.ReLU(),
            nn.Linear(self.params.d_model, self.params.actions_count),
        )

    def forward(self, x): 
        return self.network(x / 255.0)

    @staticmethod
    def _layer_init(layer, std=np.sqrt(2), bias_const=0.0):
        torch.nn.init.orthogonal_(layer.weight, std)
        torch.nn.init.constant_(layer.bias, bias_const)
        return layer

## Smoke test

In [14]:
# @launchit.disable
ap = Agent.Params(
    actions_count=10,
)
agent = Agent(ap)
agent = agent.to(CONFIG.cuda_device)
print(agent)
params_count = sum(p.numel() for p in agent.parameters())
print(f'{params_count=:_}')
probe_batch = torch.zeros((2, 4, 84, 84)).to(CONFIG.cuda_device)
print(f'{probe_batch.shape=}')
r = agent(probe_batch)
print(f'{r.shape=}')

Agent(
  (network): Sequential(
    (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=512, bias=True)
    (8): ReLU()
    (9): Linear(in_features=512, out_features=10, bias=True)
  )
)
params_count=1_689_258
probe_batch.shape=torch.Size([2, 4, 84, 84])
r.shape=torch.Size([2, 10])


## Quick play test

In [15]:
# @launchit.disable
ap = Agent.Params(
    actions_count=LS.env.action_space.n.item(),
)
agent = Agent(ap)
agent = agent.to(CONFIG.cuda_device)
env = create_env(HP.env_id, is_auto_reset=True)
obs, _ = env.reset(seed=HP.random_seed)
device = next(iter(agent.parameters())).device

with eval_guard(agent):
    with torch.no_grad():
        for step in tqdm(range(0, 1000)): 
            obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(device)
            logits = agent(obs)[0]
            action = torch.argmax(logits).cpu().item()
            obs, reward, terminated, truncated, info = env.step(action)

            if terminated or truncated:
                if env.get_wrapper_attr('was_real_done'):
                    assert 'episode' in info
                    
                    if 'episode' in info: # 'episode' is a default stats_key for RecordEpisodeStatistics
                        episode_stats = info['episode']
                        print(f'{step:05}', episode_stats)
                else:
                    assert not 'episode' in info
            else:
                assert not 'episode' in info

  0%|          | 0/1000 [00:00<?, ?it/s]

00118 {'r': 0.0, 'l': 524, 't': 0.303398}
00238 {'r': 0.0, 'l': 542, 't': 0.2748}
00358 {'r': 0.0, 'l': 531, 't': 0.266808}
00478 {'r': 0.0, 'l': 535, 't': 0.267998}
00598 {'r': 0.0, 'l': 514, 't': 0.264129}
00718 {'r': 0.0, 'l': 528, 't': 0.267013}
00838 {'r': 0.0, 'l': 524, 't': 0.265426}
00958 {'r': 0.0, 'l': 540, 't': 0.26856}


## Configure

In [16]:
# @launchit.disable
# @launchit.collect_1
# ...
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': None,
 'global_steps_count': None,
 'warmup_steps_count': None,
 'learn_steps_period': None,
 'oracle_steps_period': None,
 'start_e': None,
 'end_e': None,
 'exploration_fraction': None,
 'replay_buffer_size': None,
 'tau': None,
 'capture_video': None,
 'batch_size': None,
 'optimizer': None,
 'learn_rate': None}


## Create

In [ ]:
# @launchit.disable_2
ap = Agent.Params(
    actions_count=LS.env.action_space.n.item()
)
LS.agent = Agent(ap).to(CONFIG.cuda_device)
LS.agent.share_memory() # criticial for main/child processes interactions, allows to avoid weights copy
LS.oracle = Agent(ap).to(CONFIG.cuda_device)

# Don't use load_state_dict since it may lead to LS.oracle is sticked to LS.agent's weights
with torch.no_grad():
    for oracle_param, agent_param in zip(LS.oracle.parameters(), LS.agent.parameters()):
        oracle_param.data.copy_(agent_param.data)

# Video

## create_capture_video_policy

In [18]:
# @launchit.disable_2
def create_capture_video_policy(capture_video):
    def inner_create_capture_video_policy(policy_name, period):
        if policy_name == 'every':
            def every_thunk(count):
                return (count % period) == 0

            return every_thunk
        
        assert False, f'Unsupported capture_video="{module_name}"'
            
    ump = hp_parse_universal_module(capture_video)
    return inner_create_capture_video_policy(ump.module_name, *ump.args, **ump.kwargs)

## get_fresh_video_dir_name

In [19]:
# @launchit.disable_2
def get_fresh_video_dir_name():
    lc = HP.launch_component()
    timestamp = datetime.datetime.now().strftime("%Y.%m.%d-%H:%M:%S.%f") # generate unique dir name in order to shut up RecordVideo from complaining
    return os.path.join(CONFIG.run_path, f'video-{lc.name}-launch{lc.version}-{timestamp}')

## capture_video_of_test_rollout

In [20]:
def capture_video_of_test_rollout(agent, max_steps_count=10_000, video_dir_name=None):
    video_dir_name = lu.coalesce(video_dir_name, lambda: get_fresh_video_dir_name())
    assert video_dir_name is not None
    env = create_env(HP.env_id, video_dir_name=video_dir_name, is_auto_reset=True)
    assert env.action_space.n.item() == agent.params.actions_count
    obs, _ = env.reset(seed=HP.random_seed)
    device = next(iter(agent.parameters())).device
    
    with eval_guard(agent):
        with torch.no_grad():
            for step in range(0, max_steps_count): 
                obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(device)
                logits = agent(obs)[0]
                action = torch.argmax(logits).cpu().item()
                obs, reward, terminated, truncated, info = env.step(action)
                
                if (terminated or truncated) and env.get_wrapper_attr('was_real_done'):
                    break

    game_meta = dict(
        reward=env.get_wrapper_attr('episode_returns'),
        frames_count=env.get_wrapper_attr('episode_lengths'),
        steps_count=step + 1,
    )
    
    env.close() # this forces video recording to complete and write video file
    
    video_fnames = list(filter(lambda fn: os.path.isfile(os.path.join(video_dir_name, fn)), os.listdir(video_dir_name)))
    assert len(video_fnames) == 1, len(video_fnames)
    video_fname = os.path.join(video_dir_name, video_fnames[0])
    video_meta = {}
    
    with open(video_fname, 'rb') as f:
        container = av.open(f)
        video_meta['fps'] = float(container.streams.video[0].average_rate)
        video_stream = container.streams.video[0]
        video_meta['duration'] = float(video_stream.duration * video_stream.time_base)
        
    return video_fname, dict(game=game_meta, video=video_meta)

In [21]:
# @launchit.disable
capture_video_of_test_rollout(LS.agent, max_steps_count=2000)

('/home/misha/dev/mine/neurolab/run/17_rl/video-17b_dqn_atari_mp_01-launch0-2026.05.01-17:57:22.746352/rl-video-episode-0.mp4',
 {'game': {'reward': 0.0, 'frames_count': 524, 'steps_count': 119},
  'video': {'fps': 30.0, 'duration': 17.5}})

# Worker

Implementation details regarding memory sharing between PyTorch applications: <a href="./dialogs/torch-multiprocessing-shm.ipynb">torch-multiprocessing-shm.ipynb</a>

## StepInfo

In [22]:
StepInfo = namedtuple(
    'StepInfo', 
    'worker_ind, scoreboard_ind, action, reward, done, last_episode_reward, last_episode_length, stats'
)

## Scoreboard

In [23]:
# Shared storage for big data items
@dataclass(slots=True)
class Scoreboard:
    obs: object = None 
    next_obs: object = None 

    @staticmethod
    def create(size, env):
        # Use Torch tensors since it eases memory sharing between main/child processes
        scoreboard = Scoreboard()
        scoreboard.obs = torch.zeros((size,) + env.observation_space.shape)
        scoreboard.next_obs = torch.zeros((size,) + env.observation_space.shape)

        scoreboard.obs.share_memory_()
        scoreboard.next_obs.share_memory_()
        
        return scoreboard

## PolicyRolloutIpc

In [24]:
# @dataclass(slots=True)
@dataclass
class PolicyRolloutIpc:
    shared_state_lock: object = None
    running_agents_count: object = None
    agent_version: object = None # how many times agent was learned in main process
    no_running_agents_cond: object = None
    step_info_queue: object = None
    step_info_ack_queues: list = None

    # can't use dataclasses.asdict() as it fails to handle IPC objects like Lock because it tries to deepcopy them
    def asdict(self):
        return dict(
            shared_state_lock=self.shared_state_lock,
            running_agents_count=self.running_agents_count,
            agent_version=self.agent_version,
            no_running_agents_cond=self.no_running_agents_cond,
            step_info_queue=self.step_info_queue,
            step_info_ack_queues=self.step_info_ack_queues,
        )
    
    @staticmethod
    def create(workers_count):
        policy_rollout_ipc = PolicyRolloutIpc()
        policy_rollout_ipc.shared_state_lock = LS.mp_ctx.Lock()
        policy_rollout_ipc.running_agents_count = LS.mp_ctx.Value('i', 0)
        policy_rollout_ipc.agent_version = LS.mp_ctx.Value('i', 0)
        policy_rollout_ipc.no_running_agents_cond = LS.mp_ctx.Condition(policy_rollout_ipc.shared_state_lock)
        policy_rollout_ipc.step_info_queue = LS.mp_ctx.Queue()
        policy_rollout_ipc.step_info_ack_queues = [LS.mp_ctx.Queue() for _ in range(workers_count)]
        return policy_rollout_ipc

## WorkerTask

In [25]:
# Exchange data between main and child processes
@dataclass(slots=True)
class WorkerTask:
    task_id: int
    op: str
    params: dict = None

@dataclass(slots=True)
class WorkerTaskResult:
    task_id: int
    payload: object = None

## WorkerCtl

In [26]:
# Master's stuff (main process)
class WorkerCtl:
    task_id = 0
    
    def __init__(self, worker_ind, module, mp_ctx, scoreboard_ipc):
        self.task_ctor = getattr(module, 'WorkerTask')
        self.worker_ind = worker_ind
        self.task_queue = mp_ctx.Queue()
        self.result_queue = mp_ctx.Queue()
        self.process = mp_ctx.Process(
            target=getattr(module, 'worker_loop'), 
            args=(
                worker_ind, 
                self.task_queue, 
                self.result_queue, 
                scoreboard_ipc,
            ),
        )
        self.process.start()
        self.pending_task_ids = deque()

    @staticmethod
    def gen_task_id():
        WorkerCtl.task_id += 1
        return WorkerCtl.task_id

    def healthcheck(self):
        task = self.task_ctor(task_id=self.gen_task_id(), op='HEALTHCHECK')
        self.task_queue.put(task)
        self.result_queue.get()
        
    def terminate(self, timeout=None):
        if self.process.is_alive():
            task = self.task_ctor(task_id=self.gen_task_id(), op='TERMINATE')
            try:
                self.task_queue.put(task)
                self.result_queue.get(timeout=timeout)
                self.process.join()
            except:
                self.process.terminate()

    def init_agent(self, agent_params, agent_state_dict=None, device=None):
        if agent_state_dict is not None:
            for key in agent_state_dict:
                assert agent_state_dict[key].is_shared()
        
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='INIT_AGENT', 
            params=dict(
                agent_params=dataclasses.asdict(agent_params),
                agent_state_dict=agent_state_dict,
                device=device,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def sync_agent(self, agent_state_dict):
        for key in agent_state_dict:
            assert agent_state_dict[key].is_shared()
                
        task = self.task_ctor(
            task_id=self.gen_task_id(),
            op='SYNC_AGENT',
            params=dict(agent_state_dict=agent_state_dict),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def start_policy_rollout(self, scoreboard, global_steps_count, global_step_inc):
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='START_POLICY_ROLLOUT', 
            params=dict(
                env_id=HP.env_id,
                scoreboard=dataclasses.asdict(scoreboard),
                global_steps_count=global_steps_count,
                global_step_inc=global_step_inc,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def capture_video_of_test_rollout(self, max_steps_count, video_dir_name, forward_data=None):
        task = self.task_ctor(
            task_id=self.gen_task_id(),
            op='CAPTURE_VIDEO',
            params=dict(
                max_steps_count=max_steps_count, 
                video_dir_name=video_dir_name,
                forward_data=forward_data,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def is_busy(self):
        return len(self.pending_task_ids) > 0
    
    def get_task_result(self):
        assert len(self.pending_task_ids) > 0
        result = self.result_queue.get()
        assert result.task_id == self.pending_task_ids.popleft()
        return result

    def peek_task_result(self):
        if not self.pending_task_ids:
            return None
        
        try:
            result = self.result_queue.get(block=False)
            assert result.task_id == self.pending_task_ids.popleft()
            return result
        except queue.Empty:
            return None # task is not completed yet

    def drain_task_results(self):
        results = []

        while self.pending_task_ids:
            task_id = self.pending_task_ids.popleft()
            task_result = self.result_queue.get()
            assert task_result.task_id == task_id
            results.append(task_result)

        return results

## worker_loop

In [27]:
# Executed in child process. 
# Eager mode - worker constantly executes policy and produces actions, values, etc which are advertised to main process via scoreboard. 
# We use GPU shared agent's weights, there is no copy or the like - just SharedRW lock to ensure that we always use consistent weights (no halfupdated)
# In order to keep up with main process there are queues for free and busy indices within scoreboard. Produced item is placed into scoreboard
# and corresponding slot index is put into scoreboard_ready_inds_queue. Once this produced item is consumed by main process it returns slot index via scoreboard_free_inds_queue
def worker_loop(worker_ind, task_queue, task_result_queue, policy_rollout_ipc):
    class RunMode(Enum):
        STANDBY = auto(), 60
        POLICY_ROLLOUT = auto(), 0

        def task_wait_timeout(self):
            return self.value[1]
    
    @dataclass(slots=True)
    class WorkerState:
        run_mode: object = None
        agent: object = None
        is_attached_agent: bool = False
        # POLICY_ROLLOUT mode related attrs
        policy_rollout_ipc: object = None
        scoreboard: object = None
        scoreboard_free_inds: set = None
        agent_version: int = 0
        agent_version_steps_count: int = 0
        avg_steps_count_per_agent_version: object = None
        env: object = None
        obs: object = None
        obs_for_inference: object = None
        action: object = None
        global_steps_count: int = 0
        global_step: int = 0
        global_step_inc: int = 0
    
    CONFIG = create_config()
    LOG = Logging.get()
    LOG.app_name = CONFIG.self_name
    LOG.enable('syslog', True)
    LOG.enable('stdout', False)
    worker_random_seed = HP.random_seed + worker_ind
    
    with LOG.auto_prefix('WRK', worker_ind, 'SEED', worker_random_seed):
        LOG(f'CONFIG={CONFIG._asdict()}')
        
        au.init()
        random.seed(worker_random_seed)
        torch.manual_seed(worker_random_seed)
        RNG = np.random.default_rng(worker_random_seed)
        LOG(f'{worker_random_seed=}')
        
        torch.backends.cudnn.deterministic = HP.torch_deterministic
        LOG(f'{torch.backends.cudnn.deterministic=}')

        WS = WorkerState()
        WS.run_mode = RunMode.STANDBY
        WS.policy_rollout_ipc = lu.when(policy_rollout_ipc, lambda: PolicyRolloutIpc(**policy_rollout_ipc), None)
        LOG('Worker is ready')
        
        is_running = True
        
        while is_running:
            try:
                # task is expected to be an instanace of WorkerTask class
                task = task_queue.get(block=True, timeout=WS.run_mode.task_wait_timeout())
            except queue.Empty:
                if WS.run_mode == RunMode.STANDBY:
                    LOG(f'Didn\'t get any tasks within {WS.run_mode.task_wait_timeout()} seconds, waiting again')
                    continue
                else:
                    # In POLICY_ROLLOUT mode we just go to non-stop policy rollout
                    task = None

            if task is not None:
                with LOG.auto_prefix('TASK', task.task_id):
                    LOG(f'Got task #{task.task_id} {task.op}')
                    task_result = WorkerTaskResult(task_id=task.task_id)
                    
                    match task.op:
                        case 'HEALTHCHECK':
                            pass
                        case 'TERMINATE':
                            is_running = False
                        case 'INIT_AGENT':
                            ap = Agent.Params(**task.params['agent_params'])
                            device = lu.coalesce(task.params.get('device'), CONFIG.cuda_device)
                            WS.agent = Agent(ap).to(device)
                            WS.is_attached_agent = task.params['agent_state_dict'] is not None
                            
                            if WS.is_attached_agent:
                                # Storage for weights of attached agent is pointed to agent's weights in main process
                                state_dict = task.params['agent_state_dict']
                                
                                with torch.no_grad():
                                    for name, param in WS.agent.named_parameters():
                                        assert state_dict[name].is_shared()
                                        param.data = state_dict[name]

                            LOG(f'{WS.is_attached_agent=}')
                        case 'SYNC_AGENT':
                            assert WS.agent is not None
                            assert not WS.is_attached_agent
                            
                            state_dict = task.params['agent_state_dict']
                            
                            with torch.no_grad():
                                # Fast GPU-TO-GPU sync
                                for name, param in WS.agent.named_parameters():
                                    assert state_dict[name].is_shared()
                                    param.copy_(state_dict[name])
                        case 'START_POLICY_ROLLOUT':
                            WS.scoreboard = Scoreboard(**task.params['scoreboard'])
                            WS.scoreboard_free_inds = set(range(len(WS.scoreboard.obs)))
                            LOG(f'Scoreboard initialized: {len(WS.scoreboard.obs)=}')
                                                          
                            WS.env = create_env(task.params['env_id'], random_seed=worker_random_seed, is_auto_reset=True)
                            assert isinstance(WS.env.action_space, gym.spaces.Discrete), "only discrete action space is supported"
                            LOG(f'Env created: {WS.env.observation_space=}, {WS.env.action_space=}, {WS.env.metadata=}')
    
                            WS.obs, _ = WS.env.reset(seed=worker_random_seed)
                            WS.obs = torch.tensor(WS.obs)
                            WS.obs_for_inference = WS.obs.unsqueeze(0).to(CONFIG.cuda_device, non_blocking=True)
                            WS.action = None
                            WS.agent_version = 0
                            WS.agent_version_steps_count = 0
                            WS.avg_steps_count_per_agent_version = RecursiveAverageFilter()
                            WS.global_steps_count = task.params['global_steps_count']
                            WS.global_step = 0
                            WS.global_step_inc = task.params['global_step_inc']
                            LOG('Env reset')
                            
                            WS.run_mode = RunMode.POLICY_ROLLOUT
                        case 'CAPTURE_VIDEO':
                            assert WS.agent is not None
                        
                            with torch.no_grad():
                                video_fname, video_meta = capture_video_of_test_rollout(
                                    WS.agent, 
                                    max_steps_count=task.params['max_steps_count'], 
                                    video_dir_name=task.params['video_dir_name'])
                                task_result.payload = (video_fname, video_meta, task.params['forward_data'])
                        case _:
                            LOG(f'Unknown {task.op=}, ignoring')
            
                    task_result_queue.put(task_result)
                    LOG('Task complete')

                continue

            assert task is None
            assert WS.run_mode == RunMode.POLICY_ROLLOUT
            assert WS.agent is not None
            assert WS.env is not None
            assert WS.scoreboard is not None
            assert WS.policy_rollout_ipc is not None

            if WS.action is None:
                def linear_schedule(start_e: float, end_e: float, duration: int, t: int):
                    slope = (end_e - start_e) / duration
                    return max(slope * t + start_e, end_e)
                    
                random_action_thres = linear_schedule(HP.start_e, HP.end_e, HP.exploration_fraction * WS.global_steps_count, WS.global_step)
                    
                with WS.policy_rollout_ipc.shared_state_lock:
                    WS.policy_rollout_ipc.running_agents_count.value += 1
        
                if RNG.random() < random_action_thres:
                    WS.action = WS.env.action_space.sample()
                else:
                    logits = WS.agent(WS.obs_for_inference)[0]
                    WS.action = torch.argmax(logits).cpu().item()

                    if CONFIG.is_cuda:
                        torch.cuda.synchronize() # make sure inference kernel completed so we don't overlap with agent's update

                with WS.policy_rollout_ipc.shared_state_lock:
                    WS.policy_rollout_ipc.running_agents_count.value -= 1
                    assert WS.policy_rollout_ipc.running_agents_count.value >= 0
                    new_agent_version = WS.policy_rollout_ipc.agent_version.value

                    if WS.policy_rollout_ipc.running_agents_count.value == 0:
                        WS.policy_rollout_ipc.no_running_agents_cond.notify()
            else:
                # this means that we didn't advertise previous results because we are out of free space in scoreboard and because main process is busy
                pass

            if not WS.scoreboard_free_inds:
                ack_queue = WS.policy_rollout_ipc.step_info_ack_queues[worker_ind]
                
                try:
                    free_ind = ack_queue.get(block=True, timeout=1) # wait until free inds arrive
                except queue.Empty:
                    continue

                while True:
                    try:
                        WS.scoreboard_free_inds.add(free_ind)
                        free_ind = ack_queue.get(block=False) # if there are pending any then grab them all one be one
                    except queue.Empty:
                        break

            assert len(WS.scoreboard_free_inds) > 0
            assert WS.action is not None
            action = WS.action
            WS.action = None
            scoreboard_ind = WS.scoreboard_free_inds.pop()

            # Consume pending action by making a STEP and advertise results
            obs = WS.obs
            next_obs, reward, terminated, truncated, info = WS.env.step(action)
            
            # Immediately start GPU data transfer for next Agent inference
            WS.obs = torch.tensor(next_obs)
            WS.obs_for_inference = WS.obs.unsqueeze(0).to(CONFIG.cuda_device, non_blocking=True)

            # Advertise step info to main process
            WS.scoreboard.obs[scoreboard_ind] = obs
            WS.scoreboard.next_obs[scoreboard_ind] = WS.obs
                                                
            if WS.agent_version != new_agent_version:
                assert WS.agent_version < new_agent_version
                WS.avg_steps_count_per_agent_version(WS.agent_version_steps_count)
                WS.agent_version = new_agent_version
                WS.agent_version_steps_count = 0

                # if worker_ind == 0:
                #     for name, param in WS.agent.named_parameters():
                #         LOG(f'kms@ child: {name} -> {param.mean()}')
            else:
                WS.agent_version_steps_count += 1
                
            step_info = StepInfo(
                worker_ind=worker_ind,
                scoreboard_ind=scoreboard_ind,
                action=action,
                reward=reward,
                done=terminated or truncated,
                last_episode_reward=lu.coalesce(info.get('episode', {})).get('r', None),
                last_episode_length=lu.coalesce(info.get('episode', {})).get('l', None),
                stats=dict(
                    avg_steps_count_per_agent_version=WS.avg_steps_count_per_agent_version.v,
                ),
            )
            WS.policy_rollout_ipc.step_info_queue.put(step_info)
            WS.global_step += WS.global_step_inc
        
        LOG('Worker is going down')

## get_worker_factory

In [28]:
def get_worker_factory():
    with LOG.auto_log_level(logging.INFO):
        expandvars = dict(
            PROJECT_ROOT_PATH=CONFIG.project_root_path,
            MODEL_NAME=CONFIG.self_name,
            MODEL_VERSION=HP.launch_component().version,
            LAUNCH_GOAL=LaunchGoal.WORKER.value,
        )
        module_fname = launchit.launchit(
            CONFIG.self_fname, 
            expandvars=expandvars, 
            make_py_file=True,
            dir_name=CONFIG.run_path,
            disable_inds=[2]
        )
        LOG.info(f'Created "{module_fname}"')
        
    module_dir_name = os.path.dirname(module_fname)
    module_name = os.path.splitext(os.path.basename(module_fname))[0]
    sys.path.append(module_dir_name)
    module = __import__(module_name)

    def factory(worker_ind, rollout_policy_ipc):
        return WorkerCtl(worker_ind, module, LS.mp_ctx, rollout_policy_ipc)

    return factory

## Test

### start_rollout_policy

In [29]:
# @launchit.disable
test_workers_count = 4

env = create_env(HP.env_id)
test_scoreboards = [Scoreboard.create(10, env) for i in range(test_workers_count)]
policy_rollout_ipc = PolicyRolloutIpc.create(test_workers_count)
wf = get_worker_factory()
test_workers = [wf(i, policy_rollout_ipc.asdict()) for i in range(test_workers_count)]
test_worker_to_avg_steps_count_per_agent_version = defaultdict(int)

try:
    for w in test_workers: 
        w.init_agent(LS.agent.params, LS.agent.state_dict()) # create attached agent
        w.drain_task_results()
    
    test_steps_count = 50 * 100 * len(test_workers)
    
    for w, sb in zip(test_workers, test_scoreboards):
        w.start_policy_rollout(sb, global_steps_count=test_steps_count, global_step_inc=len(test_workers))
    
    ready_buffer = []
    
    for step in tqdm(range(test_steps_count)):
        step_info = StepInfo(*policy_rollout_ipc.step_info_queue.get(timeout=1))
        test_worker_to_avg_steps_count_per_agent_version[step_info.worker_ind] = step_info.stats['avg_steps_count_per_agent_version']
        ready_buffer.append((step_info.worker_ind, step_info.scoreboard_ind))

        if len(ready_buffer) > 32:
            with policy_rollout_ipc.no_running_agents_cond:
                while policy_rollout_ipc.running_agents_count.value > 0:
                    policy_rollout_ipc.no_running_agents_cond.wait()

                assert policy_rollout_ipc.running_agents_count.value == 0
                time.sleep(0.02) # emulate agent version update
                policy_rollout_ipc.agent_version.value += 1 # this value is used by child processes to track how many steps their made between agent's weights' updates
                
            for ready_buffer_item in ready_buffer:
                policy_rollout_ipc.step_info_ack_queues[ready_buffer_item[0]].put(ready_buffer_item[1])
                
            ready_buffer = []

        if (step % 100) == 0:
            avg_log_string = []
            
            for i in range(len(test_workers)):
                avg_log_string.append(f'{i}={test_worker_to_avg_steps_count_per_agent_version[i]:.2f}')
    
            print(','.join(avg_log_string) + ' ' * 20, end='\r')
finally:
    for w in test_workers: 
        w.terminate(timeout=3)

Created "/home/misha/dev/mine/neurolab/run/17_rl/17b_dqn_atari_mp_01-launch99.py"


  0%|          | 0/20000 [00:00<?, ?it/s]

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


0=7.27,1=7.02,2=7.38,3=7.34                    

### capture_video_of_test_rollout

In [30]:
# @launchit.disable
wf = get_worker_factory()
test_worker = wf(0, None)
try:
    test_worker.init_agent(LS.agent.params)
    test_worker.get_task_result()
    
    video_dir_name = get_fresh_video_dir_name()
    tid = test_worker.sync_agent(LS.agent.state_dict())
    test_worker.capture_video_of_test_rollout(max_steps_count=2000, video_dir_name=video_dir_name)
    tr = test_worker.get_task_result() # wait for weights are synced is done
    assert tr.task_id == tid
    result = test_worker.drain_task_results()[-1]
    print(result)
finally:
    test_worker.terminate()

Created "/home/misha/dev/mine/neurolab/run/17_rl/17b_dqn_atari_mp_01-launch100.py"


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


WorkerTaskResult(task_id=15, payload=('/home/misha/dev/mine/neurolab/run/17_rl/video-17b_dqn_atari_mp_01-launch0-2026.05.01-17:58:07.701888/rl-video-episode-0.mp4', {'game': {'reward': 0.0, 'frames_count': 524, 'steps_count': 119}, 'video': {'fps': 30.0, 'duration': 17.5}}, None))


## Configure

In [31]:
# @launchit.disable
# @launchit.collect_1
# ...
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': None,
 'global_steps_count': None,
 'warmup_steps_count': None,
 'learn_steps_period': None,
 'oracle_steps_period': None,
 'start_e': None,
 'end_e': None,
 'exploration_fraction': None,
 'replay_buffer_size': None,
 'tau': None,
 'capture_video': None,
 'batch_size': None,
 'optimizer': None,
 'learn_rate': None}


## Create

In [32]:
# @launchit.disable_2
wf = get_worker_factory()
LS.policy_rollout_ipc = PolicyRolloutIpc.create(HP.envs_count)
LS.workers = [wf(i, LS.policy_rollout_ipc.asdict()) for i in range(HP.envs_count)]

# Init in serial, shows to be much faster than in parallel (GPU contention issues?)
for w in LS.workers: 
    w.init_agent(LS.agent.params, LS.agent.state_dict())
    w.get_task_result()

LS.capture_video_worker = wf(0, None)
LS.capture_video_worker.init_agent(LS.agent.params)
LS.capture_video_worker.get_task_result()

Created "/home/misha/dev/mine/neurolab/run/17_rl/17b_dqn_atari_mp_01-launch101.py"


WorkerTaskResult(task_id=25, payload=None)

# TRAIN

## Configure

In [33]:
# @launchit.disable
# @launchit.collect

# RL params
HP.gamma = 0.99 # the discount factor gamma

# Training procedure params (DQN related) 
HP.global_steps_count = 100_000
HP.warmup_steps_count = 80_000
HP.learn_steps_period = 4 
HP.oracle_steps_period = 250 
HP.start_e = 1 
HP.end_e = 0.01 
HP.exploration_fraction = 0.1 
HP.replay_buffer_size = 100_000
HP.tau = 1 
HP.capture_video = 'every(10000)' 

# Optimization params
HP.batch_size = 32
HP.optimizer = 'Adam()'
HP.learn_rate = 1e-4

# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': 0.99,
 'global_steps_count': 100000,
 'warmup_steps_count': 80000,
 'learn_steps_period': 4,
 'oracle_steps_period': 250,
 'start_e': 1,
 'end_e': 0.01,
 'exploration_fraction': 0.1,
 'replay_buffer_size': 100000,
 'tau': 1,
 'capture_video': 'every(10000)',
 'batch_size': 32,
 'optimizer': 'Adam()',
 'learn_rate': 0.0001}


## Create

In [35]:
# @launchit.disable_2
ump = hp_parse_universal_module(HP.optimizer)
assert not ump.args
lr_params = hp_parse_learn_rate(HP.learn_rate)
optimizer = getattr(torch.optim, ump.module_name)(LS.agent.parameters(), lr=lr_params.learn_rate, **ump.kwargs)
capture_video_policy = create_capture_video_policy(HP.capture_video)
scoreboards = [Scoreboard.create(HP.learn_steps_period * 3, LS.env) for i in range(len(LS.workers))]
replay_buffer = ReplayBuffer(
    buffer_size=HP.replay_buffer_size, 
    observation_space=LS.env.observation_space,
    action_space=LS.env.action_space,
    device=CONFIG.cuda_device,
    optimize_memory_usage=True,
    handle_timeout_termination=False,
    n_envs=1, # pretend we have 1 env since we would receive scoreboard items from workers in one by one fashion
)
last_episode_rewards = np.zeros(HP.envs_count)
last_episode_lengths = np.zeros(HP.envs_count)

## Train

In [38]:
# @launchit.disable_2
class TrainState(IntEnum):
    WARMUP = auto()
    LEARN = auto()

train_state = TrainState.WARMUP
start_time = time.time()

for worker, scoreboard in zip(LS.workers, scoreboards):
    worker.start_policy_rollout(scoreboard, global_steps_count=HP.global_steps_count, global_step_inc=len(LS.workers))

for global_step in tqdm(range(HP.global_steps_count)):
    with LOG.auto_prefix('STEP', global_step):
        # TODO: handle timeout
        step_info = StepInfo(*LS.policy_rollout_ipc.step_info_queue.get(timeout=1))
        scoreboard = scoreboards[step_info.worker_ind]
        replay_buffer.add(
            obs=scoreboard.obs[step_info.scoreboard_ind].unsqueeze(0).numpy(),
            next_obs=scoreboard.next_obs[step_info.scoreboard_ind].unsqueeze(0).numpy(),
            action=np.array([step_info.action], dtype=np.float32),
            reward=np.array([step_info.reward], dtype=np.float32),
            done=np.array([step_info.done], dtype=np.float32), 
            infos=[],
        )
        last_episode_rewards[step_info.worker_ind] = lu.coalesce(step_info.last_episode_reward, last_episode_rewards[step_info.worker_ind])
        last_episode_lengths[step_info.worker_ind] = lu.coalesce(step_info.last_episode_length, last_episode_lengths[step_info.worker_ind])
        LS.policy_rollout_ipc.step_info_ack_queues[step_info.worker_ind].put(step_info.scoreboard_ind)
    
        match train_state:
            case TrainState.WARMUP:
                if global_step >= HP.warmup_steps_count:
                    train_state = TrainState.LEARN
                    
            case TrainState.LEARN:
                # Learn Agent
                if (global_step % HP.learn_steps_period) == 0:
                    data = replay_buffer.sample(HP.batch_size)
                    
                    with torch.no_grad():
                        target_max, _ = LS.oracle(data.next_observations).max(dim=1)
                        td_target = data.rewards.flatten() + HP.gamma * target_max * (1 - data.dones.flatten())
                        LOG(f'kms@ {td_target.mean().item()=}')
            
                    # data.actions.shape = [batch_size,1], gather will pick from Agent's logits values by indices=data.actions,
                    # while squeeze removes first dimension
                    old_val = LS.agent(data.observations).gather(1, data.actions).squeeze()
            
                    loss = F.mse_loss(old_val, td_target)
    
                    with LS.policy_rollout_ipc.no_running_agents_cond:
                        while LS.policy_rollout_ipc.running_agents_count.value > 0:
                            LS.policy_rollout_ipc.no_running_agents_cond.wait()
    
                        assert LS.policy_rollout_ipc.running_agents_count.value == 0
                        
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

                        if CONFIG.is_cuda:
                            torch.cuda.synchronize()
    
                        LS.policy_rollout_ipc.agent_version.value += 1 # this value is used by child processes to track how many steps their made between agent's weights' updates
            
                    summary_writer.add_scalar("losses/td_loss", loss.item(), global_step)
                    summary_writer.add_scalar("losses/q_values", old_val.mean().item(), global_step)
                    
                # update Oracle weights
                if (global_step % HP.oracle_steps_period) == 0:
                    for name, param in LS.agent.named_parameters():
                        LOG(f'kms@ main: AGENT.{name} -> {param.mean()}')

                    for name, param in LS.oracle.named_parameters():
                        LOG(f'kms@ main: ORACLE.{name} -> {param.mean()}')

                    with torch.no_grad():
                        for oracle_param, agent_param in zip(LS.oracle.parameters(), LS.agent.parameters()):
                            oracle_param.data.copy_(HP.tau * agent_param.data + (1.0 - HP.tau) * oracle_param.data)
            case _:
                assert False, f'Unsupported {train_state=}'
            
        summary_writer.add_scalar("charts/sps", int(global_step / (time.time() - start_time)), global_step)
        summary_writer.add_scalar("charts/episodic_return", last_episode_rewards.mean(), global_step)
        summary_writer.add_scalar("charts/episodic_length", last_episode_lengths.mean(), global_step)
        # TODO: report average agent's load
    
        if capture_video_policy(global_step):
            task_id = LS.capture_video_worker.sync_agent(LS.agent.state_dict())
            LS.capture_video_worker.capture_video_of_test_rollout(
                max_steps_count=5_000, 
                video_dir_name=get_fresh_video_dir_name(), 
                forward_data=dict(global_step=global_step)
            )
            task_result = LS.capture_video_worker.get_task_result() # wait until sync weights is finished so we capture video on agent with weights as LS.agent
            assert task_id == task_result.task_id
    
        if ptr := LS.capture_video_worker.peek_task_result():
            video_fname, video_meta, forward_data = ptr.payload
            _, video_fname_ext = os.path.splitext(video_fname)
            ts = datetime.datetime.now().strftime('%Y.%m.%d-%H:%M:%S')
            remote_video_fname = f'{ts}-{forward_data['global_step']:09}.{video_fname_ext.lstrip('.')}'
            summary_writer.add_file(video_fname, remote_video_fname)
            summary_writer.add_file(io.StringIO(json.dumps(video_meta)), remote_video_fname + '.meta')
            ref_text = f'<a href="http://tensorboard-videos:6007/{summary_writer.log_dir}/{remote_video_fname}" target="_blank">{remote_video_fname}</a>'
            summary_writer.add_text('videos', ref_text, forward_data['global_step'])
            
        summary_writer.flush()

if ptr := LS.capture_video_worker.peek_task_result():
    video_fname, video_meta, forward_data = ptr.payload
    _, video_fname_ext = os.path.splitext(video_fname)
    ts = datetime.datetime.now().strftime('%Y.%m.%d-%H:%M:%S')
    remote_video_fname = f'{ts}-{forward_data['global_step']:09}.{video_fname_ext.lstrip('.')}'
    summary_writer.add_file(video_fname, remote_video_fname)
    summary_writer.add_file(io.StringIO(json.dumps(video_meta)), remote_video_fname + '.meta')
    ref_text = f'<a href="http://tensorboard-videos:6007/{summary_writer.log_dir}/{remote_video_fname}" target="_blank">{remote_video_fname}</a>'
    summary_writer.add_text('videos', ref_text, forward_data['global_step'])
    summary_writer.flush()

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


  0%|          | 0/100000 [00:00<?, ?it/s]

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


## Save

In [ ]:
# @launchit.disable_2
artifact_registry = new_artifact_registry()
lc = HP.launch_component()

with io.BytesIO() as b:
    torch.save(LS.agent.state_dict(), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)

with io.StringIO() as b:
    json.dump(dataclasses.asdict(LS.agent.params), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)

# LaunchIt!

## TRAIN

In [9]:
# @launchit.disable
launchit_t0 = time.time()

In [12]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    artifact_registry_obj = new_artifact_registry(is_real=True)
    artifact_registry_obj.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=component_version, expandvars=expandvars, collect_inds=[1], disable_inds=[1])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=6
Creating /home/misha/dev/mine/neurolab/17_rl/17b_dqn_atari_mp_01-launch6.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/17_rl/17b_dqn_atari_mp_01-launch6.ipynb"


## Optuna (model selection)

### Templates

In [45]:
# @launchit.disable
# @launchit.collect_3
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP = Hyperparameters()
            HP.random_seed = 42
            assert False
        case _:
            assert False, f'Unsupported {study_serial=}'            

### Unleash

In [ ]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL:
            return ['minimize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 1
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 1
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=4, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")